# Phase 4: Feature Engineering — Telco Customer Churn

## 1. Objectives & Checklist
In **Phase 4**, we engineer domain-specific predictive features, prevent lookahead bias (data leakage), and construct a reusable preprocessing pipeline for machine learning.

### Verification Checklist:
- [x] **`Churn` converted to 0/1**: Verified as binary integer target.
- [x] **`customerID` removed**: Dropped to avoid arbitrary index memorization.
- [x] **`TotalCharges` converted to numeric**: In float64 format.
- [x] **Missing `TotalCharges` handled**: Imputed with `0.0` for `tenure == 0` accounts.
- [x] **`TotalServices` created**: Count of all active service subscriptions.
- [x] **`TenureGroup` created**: Customer lifecycle cohorts (12-month bins).
- [x] **`MonthlyChargeGroup` created**: Binned spending tiers ('Low', 'Medium', 'High', 'Very High').
- [x] **`AverageMonthlySpend` created**: Historical monthly expenditure run-rate (`TotalCharges / tenure`).
- [x] **`ChargeDifference` created**: Current MonthlyCharges relative to historical average spend.
- [x] **`IsMonthToMonth` created**: Binary flag for high-attrition contract type.
- [x] **`SupportServices` created**: Count of key protective services (`OnlineSecurity` + `TechSupport`).
- [x] **No future information used**: Strict observation-time computation (no data leakage).
- [x] **No arbitrary features created**: Every feature directly derived from domain knowledge and EDA findings.

In [1]:
# 2. Environment Setup & Data Loading
import pandas as pd
import numpy as np
from pathlib import Path

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load cleaned dataset
data_path = Path('../data/processed/cleaned_churn_data.csv')
df = pd.read_csv(data_path)
print(f"Loaded cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns.")

Loaded cleaned dataset: 7043 rows, 21 columns.


## 3. Checklist Items 1–4: Baseline Cleanliness & Identifier Handling
- Ensure `Churn` is binary integer.
- Remove `customerID` from the feature matrix.
- Confirm `TotalCharges` is clean numeric without missing values.

In [2]:
# Check Churn values
assert set(df['Churn'].unique()) == {0, 1}, "Churn is not binary 0/1!"
print(f"[x] Churn encoded: {df['Churn'].value_counts().to_dict()}")

# Remove customerID
if 'customerID' in df.columns:
    df_model = df.drop(columns=['customerID'])
    print("[x] customerID removed successfully.")
else:
    df_model = df.copy()

# Verify TotalCharges numeric
assert df_model['TotalCharges'].dtype == 'float64', "TotalCharges is not float64!"
assert df_model['TotalCharges'].isnull().sum() == 0, "Missing values in TotalCharges!"
print("[x] TotalCharges verified: float64 with 0 nulls.")

[x] Churn encoded: {0: 5174, 1: 1869}
[x] customerID removed successfully.
[x] TotalCharges verified: float64 with 0 nulls.


## 4. Checklist Items 5–11: Engineering Domain Features

### Feature Formulations:
1. **`TotalServices`**: Sum of all subscribed telephony, internet, and add-on services (range 1–9).
2. **`TenureGroup`**: Cohorts binned into 12-month intervals (`0-12`, `13-24`, `25-36`, `37-48`, `49-60`, `61-72`).
3. **`MonthlyChargeGroup`**: Spending tiers (`Low` $\le \$35$, `Medium` $\$35–\$70$, `High` $\$70–\$90$, `Very High` $> \$90$).
4. **`AverageMonthlySpend`**: Historical monthly spending rate ($\text{TotalCharges} / \text{tenure}$; imputed with $\text{MonthlyCharges}$ when $\text{tenure}=0$).
5. **`ChargeDifference`**: Difference between current billing and historical average ($\text{MonthlyCharges} - \text{AverageMonthlySpend}$). Indicates price increases or plan upgrades.
6. **`IsMonthToMonth`**: Binary indicator ($1$ if `Contract == 'Month-to-month'`, else $0$).
7. **`SupportServices`**: Count of key protective assistance features (`OnlineSecurity` + `TechSupport`).

In [3]:
# 1. TotalServices
service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_model['TotalServices'] = (df_model[service_cols] == 'Yes').sum(axis=1) + (df_model['InternetService'] != 'No').astype(int)
print("[x] TotalServices created:")
print(df_model['TotalServices'].value_counts().sort_index())

# 2. TenureGroup
df_model['TenureGroup'] = pd.cut(
    df_model['tenure'],
    bins=[-1, 12, 24, 36, 48, 60, 72],
    labels=['0-12', '13-24', '25-36', '37-48', '49-60', '61-72']
).astype(str)
print("\n[x] TenureGroup created:")
print(df_model['TenureGroup'].value_counts().sort_index())

# 3. MonthlyChargeGroup
df_model['MonthlyChargeGroup'] = pd.cut(
    df_model['MonthlyCharges'],
    bins=[0, 35, 70, 90, 150],
    labels=['Low', 'Medium', 'High', 'Very High']
).astype(str)
print("\n[x] MonthlyChargeGroup created:")
print(df_model['MonthlyChargeGroup'].value_counts().sort_index())

# 4. AverageMonthlySpend
df_model['AverageMonthlySpend'] = np.where(
    df_model['tenure'] > 0,
    df_model['TotalCharges'] / df_model['tenure'],
    df_model['MonthlyCharges']
).round(2)
print("\n[x] AverageMonthlySpend created (summary):")
print(df_model['AverageMonthlySpend'].describe().round(2))

# 5. ChargeDifference
df_model['ChargeDifference'] = (df_model['MonthlyCharges'] - df_model['AverageMonthlySpend']).round(2)
print("\n[x] ChargeDifference created (summary):")
print(df_model['ChargeDifference'].describe().round(2))

# 6. IsMonthToMonth
df_model['IsMonthToMonth'] = (df_model['Contract'] == 'Month-to-month').astype(int)
print("\n[x] IsMonthToMonth created:")
print(df_model['IsMonthToMonth'].value_counts().to_dict())

# 7. SupportServices
df_model['SupportServices'] = (df_model['OnlineSecurity'] == 'Yes').astype(int) + (df_model['TechSupport'] == 'Yes').astype(int)
print("\n[x] SupportServices created:")
print(df_model['SupportServices'].value_counts().to_dict())

[x] TotalServices created:
TotalServices
1    1264
2     859
3     846
4     965
5     922
6     908
7     676
8     395
9     208
Name: count, dtype: int64

[x] TenureGroup created:
TenureGroup
0-12     2186
13-24    1024
25-36     832
37-48     762
49-60     832
61-72    1407
Name: count, dtype: int64

[x] MonthlyChargeGroup created:
MonthlyChargeGroup
High         1844
Low          1735
Medium       1725
Very High    1739
Name: count, dtype: int64

[x] AverageMonthlySpend created (summary):
count    7043.00
mean       64.76
std        30.19
min        13.78
25%        35.94
50%        70.34
75%        90.17
max       121.40
Name: AverageMonthlySpend, dtype: float64

[x] ChargeDifference created (summary):
count    7043.00
mean       -0.00
std         2.61
min       -18.90
25%        -1.16
50%         0.00
75%         1.14
max        19.12
Name: ChargeDifference, dtype: float64

[x] IsMonthToMonth created:
{1: 3875, 0: 3168}

[x] SupportServices created:
{0: 4079, 1: 1865, 2: 1099}


## 5. Statistical Association of Engineered Features with Churn
Validating that each engineered feature exhibits strong association with the target.

In [4]:
engineered_features = ['IsMonthToMonth', 'SupportServices', 'TenureGroup', 'MonthlyChargeGroup', 'TotalServices']
for feat in engineered_features:
    assoc = df_model.groupby(feat)['Churn'].agg(
        Count='count',
        Churn_Rate_Pct=lambda x: (x.mean() * 100).round(2)
    )
    print(f"=== CHURN RATE BY {feat} ===")
    print(assoc)
    print()

=== CHURN RATE BY IsMonthToMonth ===
                Count  Churn_Rate_Pct
IsMonthToMonth                       
0                3168            6.76
1                3875           42.71

=== CHURN RATE BY SupportServices ===
                 Count  Churn_Rate_Pct
SupportServices                       
0                 4079           33.42
1                 1865           21.82
2                 1099            9.01

=== CHURN RATE BY TenureGroup ===
             Count  Churn_Rate_Pct
TenureGroup                       
0-12          2186           47.44
13-24         1024           28.71
25-36          832           21.63
37-48          762           19.03
49-60          832           14.42
61-72         1407            6.61

=== CHURN RATE BY MonthlyChargeGroup ===
                    Count  Churn_Rate_Pct
MonthlyChargeGroup                       
High                 1844           37.80
Low                  1735           10.89
Medium               1725           23.94
Very High 

## 6. Preprocessing Pipeline Construction (`ColumnTransformer`)
Setting up standard scaling for numeric features and one-hot encoding for categorical features.

In [5]:
import sys
sys.path.append('..')
from src.feature_engineering import get_feature_types, build_preprocessor

num_cols, cat_cols = get_feature_types(df_model)
print(f"Numeric features ({len(num_cols)}): {num_cols}")
print(f"\nCategorical features ({len(cat_cols)}): {cat_cols}")

preprocessor = build_preprocessor(num_cols, cat_cols)
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_preprocessed = preprocessor.fit_transform(X)
print(f"\nPreprocessed feature matrix shape: {X_preprocessed.shape[0]} rows x {X_preprocessed.shape[1]} columns.")

Numeric features (9): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'AverageMonthlySpend', 'ChargeDifference', 'IsMonthToMonth', 'SupportServices']

Categorical features (17): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup', 'MonthlyChargeGroup']

Preprocessed feature matrix shape: 7043 rows x 36 columns.


## 7. Verification with Production Module (`src.feature_engineering`)
Confirm that `src.feature_engineering.engineer_features` produces identical outputs.

In [6]:
from src.feature_engineering import engineer_features
module_df_feat = engineer_features(df)

pd.testing.assert_frame_equal(df_model, module_df_feat)
print("SUCCESS: Production module src.feature_engineering.engineer_features matches perfectly!")

SUCCESS: Production module src.feature_engineering.engineer_features matches perfectly!
